<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_25_fine_tuning_examples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🧪 **Module 25 — Fine-Tuning Examples (Full · LoRA · QLoRA · SFT with TRL)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)


# Module 25 — Fine-Tuning Examples

> **Premise:** in M17 you saw `pipeline("sentiment-analysis")` — load any pretrained model in 3 lines. But what if no model on Hugging Face does *exactly* what you need? You **fine-tune** one. This module shows the four ways modern teams do it, with runnable examples that fit on free Colab.

### What you'll cover
1. The fine-tuning landscape — full vs LoRA vs QLoRA vs SFT
2. When to fine-tune (vs prompt engineering vs RAG)
3. **Example 1:** Full fine-tuning a small model on a custom corpus
4. **Example 2:** LoRA — same task, much smaller adapter, much faster
5. **Example 3:** Save and load the LoRA adapter
6. **Example 4:** SFT with TRL — the modern way for instruction-tuning
7. Where this scales — DPO, RLHF, Constitutional AI

> ⏱ **Runtime:** ~5 minutes on a Colab T4 GPU; ~15 minutes on Colab CPU. Reduce `max_steps` in each cell if needed.


## 0 · Setup

In [ ]:
!pip -q install -U transformers datasets peft trl accelerate bitsandbytes evaluate

import os, torch, numpy as np, random
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          TrainingArguments, Trainer)

def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch:", torch.__version__)

## 1 · The Fine-Tuning Landscape

| Method | What changes | Memory | Speed | When to use |
|---|---|---|---|---|
| **Full fine-tuning** | every weight | huge — N × forward | slow | small model, lots of GPU, custom domain |
| **LoRA** | tiny low-rank adapters next to each weight matrix | ~1% of full | fast | the modern default for most fine-tuning |
| **QLoRA** | LoRA on top of a 4-bit quantised base model | runs 70B on 1× 4090 | medium | huge models on small hardware |
| **SFT** (Supervised Fine-Tuning) | usually LoRA, but on instruction-response pairs | as LoRA | as LoRA | teaching a base LM to follow instructions |
| **DPO / RLHF** | preference alignment after SFT | varies | medium | aligning a model to human preferences |

### When to fine-tune at all

Most "I want the model to do X" tasks are solved cheaper by **prompt engineering** or **RAG** (retrieval-augmented generation). Fine-tune when:

- You need the model to learn a **style or format** that's hard to specify in a prompt.
- You have a **domain corpus** (medical, legal, code) the base model doesn't know well.
- You're optimising for **inference cost** — a fine-tuned 1B can beat a prompted 70B for narrow tasks.


## 2 · A Tiny Custom Dataset

We'll teach a small GPT-2 to write in the style of a fictional company's marketing voice. ~50 examples, just enough to see fine-tuning take effect.

In [ ]:
marketing_examples = [
    "Our cloud platform is built for teams that ship daily, not quarterly.",
    "Stop juggling 12 tools. Run your entire pipeline from one dashboard.",
    "Built by engineers who actually use the product they ship.",
    "Real-time collaboration, with no Slack required.",
    "Your data stays in your VPC. Always.",
    "Migrate from legacy systems in days, not months.",
    "AI-assisted, human-approved, customer-loved.",
    "Audit logs that pass SOC 2 without a fight.",
    "Single-tenant when you need it; multi-tenant when you want it.",
    "Free for solo devs. Affordable for teams. Honest pricing for enterprises.",
] * 5   # repeat 5x for a tiny but learnable signal

print(f"{len(marketing_examples)} training examples")
print("sample:", marketing_examples[0])

ds = Dataset.from_dict({"text": marketing_examples})
ds = ds.train_test_split(test_size=0.1, seed=42)
print(ds)

## 3 · Example 1 — Full Fine-Tuning

Update **every weight** in `distilgpt2` (88M params). Slow and memory-hungry, but the simplest mental model.

In [ ]:
MODEL_NAME = "distilgpt2"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
print(f"params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def tokenize(batch):
    out = tok(batch["text"], truncation=True, padding="max_length", max_length=64)
    out["labels"] = out["input_ids"].copy()    # for causal LM, labels = inputs
    return out

ds_tok = ds.map(tokenize, batched=True, remove_columns=["text"])
print(ds_tok)

In [ ]:
args = TrainingArguments(
    output_dir="/tmp/full_ft",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10,
    report_to="none",
    save_strategy="no",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["test"],
)

# Sample BEFORE training
def generate(prompt, max_new=30):
    ids = tok(prompt, return_tensors="pt").to(device)
    out = model.generate(**ids, max_new_tokens=max_new, do_sample=True,
                         top_p=0.9, temperature=0.8, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0], skip_special_tokens=True)

print("BEFORE FT:", generate("Our platform is"))
trainer.train()
print("AFTER  FT:", generate("Our platform is"))

You should see the post-training output adopt the choppy "marketing one-liner" cadence of the training data — that's fine-tuning working.

## 4 · Example 2 — LoRA (Low-Rank Adaptation)

Instead of updating every weight, **add a tiny low-rank adapter** to each attention matrix and only train those. Result: ~0.1-1% of parameters trained, almost-equivalent quality.

The math:
$$W'\,x = (W + \alpha \cdot AB)\,x \quad\text{where } A \in \mathbb{R}^{d \times r},\ B \in \mathbb{R}^{r \times d}, r \ll d$$


In [ ]:
from peft import LoraConfig, get_peft_model

# Re-load a clean base model so the LoRA adapter starts from scratch
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

lora_cfg = LoraConfig(
    r=8,                    # rank — usually 4–32
    lora_alpha=16,           # scaling
    target_modules=["c_attn"],   # which weight matrices to adapt (GPT-2 attention)
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
args = TrainingArguments(
    output_dir="/tmp/lora_ft",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=3e-4,        # LoRA tolerates much higher LR
    logging_steps=10,
    report_to="none",
    save_strategy="no",
)

trainer = Trainer(model=model, args=args,
                  train_dataset=ds_tok["train"], eval_dataset=ds_tok["test"])

print("BEFORE LoRA:", generate("Our platform is"))
trainer.train()
print("AFTER  LoRA:", generate("Our platform is"))

## 5 · Example 3 — Save & Load the LoRA Adapter

LoRA adapters are tiny (~1-10 MB) — easy to share, swap, version. The full base model stays unchanged.

In [ ]:
ADAPTER_DIR = "/tmp/lora-marketing"
model.save_pretrained(ADAPTER_DIR)

import os
size_mb = sum(os.path.getsize(os.path.join(ADAPTER_DIR, f))
              for f in os.listdir(ADAPTER_DIR) if not f.startswith(".")) / 1e6
print(f"adapter folder: {size_mb:.2f} MB")
print("contents:", os.listdir(ADAPTER_DIR))

In [ ]:
# Load the base model + apply the adapter — full pipeline in 3 lines
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
restored = PeftModel.from_pretrained(base, ADAPTER_DIR).to(device)

# Replace global `model` so generate() uses the restored one
model = restored
print("RESTORED model:", generate("Our platform is"))

## 6 · Example 4 — SFT with TRL (the modern way)

`trl` (Hugging Face's RLHF library) gives you a `SFTTrainer` that handles instruction-tuning datasets out of the box. Same engine, much less boilerplate.

Dataset format: each row has an `instruction` and a `response`. Standard for fine-tuning chat/instruct models.

In [ ]:
from trl import SFTTrainer, SFTConfig

# Tiny instruction-tuning dataset
sft_examples = [
    {"instruction": "Write a one-line marketing tagline.", "response": "Built for teams that ship daily."},
    {"instruction": "Write a one-line marketing tagline.", "response": "Honest pricing. Real engineers. Zero lock-in."},
    {"instruction": "Write a one-line marketing tagline.", "response": "Real-time collaboration without the meetings."},
    {"instruction": "Summarize SOC 2 compliance in one sentence.", "response": "Audit logs and access controls that pass third-party review without a fight."},
    {"instruction": "Write a friendly error message for a 500.", "response": "Something went wrong on our side — we have been notified and we are looking into it."},
] * 8

def format_row(row):
    return {"text": f"### Instruction:\n{row['instruction']}\n\n### Response:\n{row['response']}"}

sft_ds = Dataset.from_list(sft_examples).map(format_row)
print(sft_ds[0]["text"])

In [ ]:
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
peft_model = get_peft_model(base, lora_cfg)

sft_args = SFTConfig(
    output_dir="/tmp/sft_lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=3e-4,
    logging_steps=10,
    report_to="none",
    save_strategy="no",
    max_seq_length=128,
    packing=False,
)

sft_trainer = SFTTrainer(
    model=peft_model,
    args=sft_args,
    train_dataset=sft_ds,
)

sft_trainer.train()

In [ ]:
def chat(prompt):
    full = f"### Instruction:\n{prompt}\n\n### Response:\n"
    ids = tok(full, return_tensors="pt").to(device)
    out = peft_model.generate(**ids, max_new_tokens=40, do_sample=True,
                               top_p=0.9, temperature=0.7, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0], skip_special_tokens=True)

print(chat("Write a one-line marketing tagline."))

## 7 · Where This Scales — Beyond SFT

| Technique | Purpose | Library |
|---|---|---|
| **QLoRA** | LoRA on a 4-bit quantised base — fine-tune 70B models on a single 24GB GPU | `bitsandbytes` + `peft` |
| **DPO** (Direct Preference Optimization) | Align a model to chosen/rejected pairs without a reward model | `trl.DPOTrainer` |
| **RLHF (PPO)** | Classical 3-stage: SFT → reward model → PPO. What ChatGPT used. | `trl.PPOTrainer` |
| **Constitutional AI** | Self-critique + revision; what Claude is trained with. | research-stage |
| **ORPO** | Combines SFT + DPO into a single training run. Newer, simpler. | `trl.ORPOTrainer` |

### Quick QLoRA recipe (concept)

```python
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
base = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3-8B", quantization_config=bnb_config
)
# Then attach LoRA on top — trains in <16GB VRAM
```

### Quick DPO recipe (concept)

```python
from trl import DPOTrainer
# dataset: rows with {prompt, chosen, rejected}
trainer = DPOTrainer(model=sft_model, ref_model=sft_model_frozen,
                     train_dataset=preference_ds, args=args)
trainer.train()
```


## 8 · Practice

1. **Different style** — replace the marketing dataset with **pirate-speak** examples (~30 rows) and re-run the full fine-tuning section. Does the model learn "ahoy" / "matey" / "ye scallywag"?

2. **Compare LR** — re-run LoRA training at `learning_rate=1e-5`, `1e-4`, `3e-4`, `1e-3`. Plot final training loss for each. LoRA is famously LR-tolerant.

3. **Adapter swapping** — train two adapters on two different styles, save both, load each onto the SAME base, generate from the same prompt. Compare outputs.

4. **Eval before/after with BLEU** — pick 5 reference completions, compute BLEU between model output and reference for the base model vs your fine-tuned model.


In [ ]:
# Sketch for Practice #1 — pirate dataset
pirate = [
    "Avast ye, the cloud platform be ready, matey!",
    "Hoist the data flag, set sail for production!",
    "No scallywag shall touch yer VPC, on me word.",
    "Pieces of eight in the audit log, every transaction!",
    "Walk the plank, legacy CMS — we sail with the new!",
] * 8

ds_pirate = Dataset.from_dict({"text": pirate}).train_test_split(test_size=0.1, seed=42)
ds_pirate_tok = ds_pirate.map(tokenize, batched=True, remove_columns=["text"])
print(ds_pirate_tok)
# (then build a fresh model, attach LoRA, train as in section 4)

## Recap

✅ Pick the right fine-tuning method from the situation (full / LoRA / QLoRA / SFT / DPO).
✅ Tokenise a custom dataset for causal-LM training.
✅ Run **full fine-tuning** with `Trainer`.
✅ Run **LoRA fine-tuning** with `peft` — ~1% trainable params, much faster.
✅ Save and load LoRA adapters as small portable files.
✅ Use **`SFTTrainer`** for instruction-tuning with the standard `### Instruction / ### Response` format.
✅ Recognise where DPO, RLHF, and QLoRA fit in the alignment story.

### What this unlocks
You can now fine-tune any open-weight model from Hugging Face on your own data — small (1-3B for quick experiments) or large (70B+ via QLoRA on a single GPU). Combined with M17-M24 you have the full pipeline: PRETRAIN → FINE-TUNE → DEPLOY.
